In [1]:
### Script to collect data from WSDC Points Registry API
# by Sarah Rempel, API posts were from Connie Wang's Jan 2018 code:
#       https://github.com/conniewang3/WSDC-Project/tree/master/part1
# 2019-12-28, updated to collect names on 2021-02-27, updated to Jupyter Notebooks 2023-02-02

In [ ]:
import pandas as pd
import openpyxl, requests, pprint, time
from datetime import date
import simplejson as json

In [3]:
### Install packages
## with conda:
#import sys
#!conda install --yes --prefix {sys.prefix} simplejson
## with pip:
#import sys
#!{sys.executable} -m pip install numpy

In [4]:
API_url = 'https://points.worldsdc.com/lookup/find'

point_df = pd.DataFrame(columns = ['wsdc_id', 'level_points', 'allowed_level', 'required_level', \
                                   'event_level', 'event_name', 'event_location', 'event_date', \
                                   'points', 'result', 'role', 'first_name', 'last_name'])
today = date.today() # the collection date will be appended to the excel file so I know when I ran it


In [ ]:
# Pick the start and end numbers. Note that it's best to do 5-10k numbers at a time (takes ~1-2hr to run)
#    instead of doing the whole thing at once (which takes 4-6 hours)
start = 13758
end = 13759
# up to 21606 as of 2023-04-01
print(point_df)

Empty DataFrame
Columns: [wsdc_id, level_points, allowed_level, required_level, event_level, event_name, event_location, event_date, points, result, role, first_name, last_name]
Index: []


In [ ]:
# Loop to go through every WSDC number
for wsdc_id in range(start, end):
    try:
        response = requests.post(API_url, {'q': wsdc_id}).json()
        
        # only cases with valid WSDC IDs containing WCS placements
        if len(response) > 2 and response['placements'] != [] and 'West Coast Swing' in response['placements'].keys():
            #print('Westie confirmed.')
            westie = response['placements']['West Coast Swing']

            #Extract dancer's level information and name
            allowed_level = response['level']['allowed']
            required_level = response['level']['required']
            first_name = response['dancer']['first_name']
            last_name = response['dancer']['last_name']
            
            #Extract information on a per level and per event basis
            for event_level in westie:
                #print(event_level + " of " + str(len(westie)))
                tot_points = westie[event_level]['total_points']
                eventList = westie[event_level]['competitions']

                for i in range(len(eventList)):
                    #print(str(i) + " of " + str(len(eventList)))
                    event_name = eventList[i]['event']['name']
                    event_date = eventList[i]['event']['date']
                    event_location = eventList[i]['event']['location']
                    points = eventList[i]['points']
                    result = eventList[i]['result']
                    role = eventList[i]['role']
                    #print(role)

                    obs = {'wsdc_id': wsdc_id, 'level_points': str(event_level)+'_'+str(tot_points), \
                           'allowed_level': allowed_level, 'required_level': required_level, \
                           'event_level': event_level, 'event_name': event_name, \
                           'event_location': event_location, 'event_date': event_date, \
                           'points': points, 'result': result, 'role': role, \
                           'first_name': first_name, 'last_name': last_name}
                    
                    point_df_new_row = pd.DataFrame(obs, index = [0])
                    point_df = pd.concat([point_df, point_df_new_row], ignore_index = True)   
                    
            if wsdc_id % 2000 == 0:
                point_df.to_csv('C:\\Users\\sarah\\Documents\\Fun Things\\West Coast Swing\\WSDC Points Registry\\Point_DF_'+str(today)+'updating.csv')
            
        print('Dancer #'+str(wsdc_id)+' completed.')
        # sleep for 0.5 seconds on each for loop to let WSDC website rest
        time.sleep(0.5)
                    
                    #print('Westie #'+str(wsdc_id)+' got '+str(points)+' points as a '+role+' in '+event_date \
                    #     +'. They are level '+allowed_level+'.')
        #else:
               #print('No such westie with #' + str(wsdc_id) + ' exists.')
    except ValueError as IndexError:
        continue
print('That\'s all the westies!')

Dancer #1 completed.
Dancer #2 completed.
Dancer #3 completed.
Dancer #4 completed.
Dancer #5 completed.
Dancer #6 completed.
Dancer #7 completed.
Dancer #8 completed.
Dancer #9 completed.
Dancer #10 completed.
Dancer #11 completed.
Dancer #12 completed.
Dancer #13 completed.
Dancer #14 completed.
Dancer #15 completed.
Dancer #16 completed.
Dancer #17 completed.
Dancer #18 completed.
Dancer #19 completed.
Dancer #20 completed.
Dancer #21 completed.
Dancer #22 completed.
Dancer #23 completed.
Dancer #24 completed.
Dancer #25 completed.
Dancer #26 completed.
Dancer #27 completed.
Dancer #28 completed.
Dancer #29 completed.
Dancer #30 completed.
Dancer #31 completed.
Dancer #32 completed.
Dancer #33 completed.
Dancer #34 completed.
Dancer #35 completed.
Dancer #36 completed.
Dancer #37 completed.
Dancer #38 completed.
Dancer #39 completed.
Dancer #40 completed.
Dancer #41 completed.
Dancer #42 completed.
Dancer #43 completed.
Dancer #44 completed.
Dancer #45 completed.
Dancer #46 complete

In [ ]:
# Export dataframe to a csv file with the path shown
point_df.to_csv('C:\\Users\\sarah\\Documents\\Fun Things\\West Coast Swing\\WSDC Points Registry\\Point_DF_'+str(start)+'-'+str(end)+'_'+str(today)+'.csv')